# 용인시 근저당 안전도 분석 AI


## 1. 라이브러리

In [ ]:
!pip install wandb -q

import pandas as pd
import numpy as np
import re
import wandb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder
from google.colab import files

print('라이브러리 로드 완료')

## 2. 데이터 로드

In [ ]:
# 엑셀 파일 업로드
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f'데이터 shape: {df.shape}')
print(f'\n컬럼 목록:')
print(df.columns.tolist())
print(f'\n데이터 미리보기:')
df.head()

## 3. 컬럼명 확인 후 매핑

In [ ]:

COL_ADDRESS  = '도로명주소'    # 도로명 주소 컬럼명
COL_SIGUNGU  = '시군구'        # 시군구 컬럼명
COL_DONG     = '읍면동'        # 읍면동 컬럼명
COL_AREA     = '연면적'        # 연면적 컬럼명
COL_LAND     = '대지면적'      # 대지면적 컬럼명
COL_TYPE     = '주택유형'      # 주택유형 컬럼명
COL_YEAR     = '건축년도'      # 건축년도 컬럼명
COL_PRICE    = '거래금액'      # 거래금액 컬럼명
COL_TRADE_Y  = '거래년도'      # 거래연도 컬럼명 (없으면 거래일에서 추출)
COL_TRADE_D  = '거래일'        # 거래일 컬럼명

print('컬럼 매핑 설정 완료')
print('실제 엑셀 컬럼:', df.columns.tolist())

## 4. 전처리

In [ ]:
df2 = df.copy()

# 거래연도 추출 (컬럼 없으면 거래일에서 추출)
if COL_TRADE_Y in df2.columns:
    df2['trade_year'] = df2[COL_TRADE_Y].astype(int)
else:
    df2['trade_year'] = pd.to_datetime(df2[COL_TRADE_D]).dt.year

# 거래금액 전처리 (쉼표 제거 후 숫자 변환)
df2['price'] = df2[COL_PRICE].astype(str).str.replace(',', '').str.strip()
df2['price'] = pd.to_numeric(df2['price'], errors='coerce')

# 결측값 제거
df2 = df2.dropna(subset=['price', COL_AREA, COL_LAND, COL_YEAR])

# building_age = 거래연도 - 건축년도
df2['building_age'] = df2['trade_year'] - df2[COL_YEAR].astype(int)
df2 = df2[df2['building_age'] >= 0]  # 음수 제거

# far = log(연면적 / 대지면적)
df2['total_area'] = pd.to_numeric(df2[COL_AREA], errors='coerce')
df2['land_area']  = pd.to_numeric(df2[COL_LAND],  errors='coerce')
df2 = df2[df2['land_area'] > 0]  # 0 나누기 방지
df2['far'] = np.log(df2['total_area'] / df2['land_area'])

# house_type: 단독=0, 다가구=1
df2['house_type'] = df2[COL_TYPE].apply(lambda x: 0 if '단독' in str(x) else 1)

# gu 추출 (수지구 / 기흥구 / 처인구)
def extract_gu(addr):
    for gu in ['수지구', '기흥구', '처인구']:
        if gu in str(addr):
            return gu
    return '기타'

df2['gu'] = df2[COL_SIGUNGU].apply(extract_gu)

# dong: 빈도 낮은 동 기타 처리
dong_counts = df2[COL_DONG].value_counts()
valid_dongs = dong_counts[dong_counts >= 50].index
df2['dong'] = df2[COL_DONG].apply(lambda x: x if x in valid_dongs else '기타')

# road: 도로명에서 큰 도로명만 추출
def extract_road(addr):
    addr = str(addr)
    match = re.search(r'([가-힣]+(?:로|길))', addr)
    return match.group(1) if match else '기타'

df2['road'] = df2[COL_ADDRESS].apply(extract_road)

# 빈도 낮은 도로명 기타 처리
road_counts = df2['road'].value_counts()
valid_roads = road_counts[road_counts >= 50].index
df2['road'] = df2['road'].apply(lambda x: x if x in valid_roads else '기타')

print(f'전처리 후 데이터 수: {len(df2)}')
print(f'gu 분포:\n{df2["gu"].value_counts()}')
print(f'\ndong 종류 수: {df2["dong"].nunique()}')
print(f'road 종류 수: {df2["road"].nunique()}')

## 5. 피처 준비 및 데이터 분할

In [ ]:
# One-Hot Encoding
cat_features = ['gu', 'dong', 'road']
num_features = ['total_area', 'land_area', 'far', 'building_age', 'house_type']

df_encoded = pd.get_dummies(df2[cat_features + num_features], columns=cat_features)
df_encoded['price'] = df2['price'].values
df_encoded['trade_year'] = df2['trade_year'].values

# 시계열 기준 분할 (2019~2023 학습 / 2024~2026 검증)
train = df_encoded[df_encoded['trade_year'] <= 2023]
test  = df_encoded[df_encoded['trade_year'] >= 2024]

feature_cols = [c for c in df_encoded.columns if c not in ['price', 'trade_year']]

X_train = train[feature_cols]
y_train = train['price']
X_test  = test[feature_cols]
y_test  = test['price']

# sample_weight: 거래연도 기반
weight_map = {2015:0.2, 2016:0.25, 2017:0.3, 2018:0.35,
              2019:0.5, 2020:0.6, 2021:0.7, 2022:0.8,
              2023:0.85, 2024:0.9, 2025:1.0, 2026:1.0}
weights = train['trade_year'].map(weight_map).fillna(0.5).values

print(f'학습 데이터: {len(X_train)}개')
print(f'검증 데이터: {len(X_test)}개')
print(f'피처 수: {len(feature_cols)}개')

## 6. WandB 설정

In [ ]:
wandb.login()  # WandB 계정 로그인

## 7. 모델 학습 및 실험 추적

In [ ]:
# 실험할 파라미터 조합
param_grid = [
    {'n_estimators': 100, 'max_depth': 5},
    {'n_estimators': 100, 'max_depth': 10},
    {'n_estimators': 300, 'max_depth': 10},
    {'n_estimators': 300, 'max_depth': 15},
    {'n_estimators': 500, 'max_depth': 15},
    {'n_estimators': 500, 'max_depth': None},
]

best_r2 = -999
best_model = None
best_params = None

for params in param_grid:
    run = wandb.init(
        project='yongin-mortgage-ai',
        name=f"RF_n{params['n_estimators']}_d{params['max_depth']}",
        config=params,
        reinit=True
    )

    model = RandomForestRegressor(
        n_estimators=params['n_estimators'],
        max_depth=params['max_depth'],
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train, sample_weight=weights)

    pred = model.predict(X_test)
    mae  = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2   = r2_score(y_test, pred)

    wandb.log({'MAE': mae, 'RMSE': rmse, 'R2': r2})
    print(f"n={params['n_estimators']}, depth={params['max_depth']} | MAE={mae:,.0f} | RMSE={rmse:,.0f} | R²={r2:.4f}")

    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_params = params

    run.finish()

print(f'\n최적 파라미터: {best_params}')
print(f'최고 R²: {best_r2:.4f}')

## 8. 피처 중요도 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'NanumGothic'

importances = best_model.feature_importances_
feat_df = pd.DataFrame({'feature': feature_cols, 'importance': importances})
feat_df = feat_df.sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
plt.barh(feat_df['feature'][::-1], feat_df['importance'][::-1])
plt.xlabel('중요도')
plt.title('피처 중요도 Top 20')
plt.tight_layout()
plt.show()

## 10. 모델 저장

In [ ]:
import pickle

with open('rf_model.pkl', 'wb') as f:
    pickle.dump({'model': best_model, 'feature_cols': feature_cols}, f)

files.download('rf_model.pkl')
print('모델 저장 완료')